<a href="https://colab.research.google.com/github/SVikramraj/syncup-event-/blob/main/ENT_Clinical_Note_Extraction_5_Modules.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ENT Clinical Note Extraction — 5 Module Google Colab Notebook

This notebook loads one instruction-tuned model and provides five reusable ENT extraction modules:

1. Chief Complaint
2. Patient History
3. Examination Findings
4. Diagnosis
5. Investigations

A final integration function combines all five outputs into one JSON object.


In [1]:
!pip install -q transformers accelerate sentencepiece


In [2]:
import json
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print("Model loaded successfully.")
print("Device:", model.device)


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully.
Device: cuda:0


## Module 1 — Chief Complaint Extraction

In [3]:
def parse_chief_complaint_json(response):
    default = {"chief_complaints": []}
    try:
        data = json.loads(response)
    except json.JSONDecodeError:
        match = re.search(r'\{[\s\S]*\}', response)
        if not match:
            return default
        try:
            data = json.loads(match.group())
        except json.JSONDecodeError:
            return default

    complaints = data.get("chief_complaints", [])
    if not isinstance(complaints, list):
        complaints = [complaints]

    cleaned = []
    for item in complaints:
        if not isinstance(item, dict):
            continue
        symptom = str(item.get("symptom", "")).strip()
        laterality = str(item.get("laterality", "Not specified")).strip() or "Not specified"
        duration = str(item.get("duration", "Not specified")).strip() or "Not specified"
        if symptom:
            cleaned.append({
                "symptom": symptom,
                "laterality": laterality,
                "duration": duration
            })
    return {"chief_complaints": cleaned}


def extract_chief_complaint(clinical_text):
    system_prompt = """You are an ENT clinical documentation assistant.

Extract ONLY the Chief Complaint from the clinical text.

Rules:
- Capture symptom(s), affected side (left/right/bilateral), and duration.
- Use the patient's own symptom words where possible.
- If multiple complaints exist, list them in order of mention.
- Do not include history, examination findings, investigations, or diagnosis.
- If duration is not mentioned, use "Not specified".
- If laterality is not mentioned, use "Not specified".
- Return valid JSON only.

Schema:
{
  "chief_complaints": [
    {"symptom": "", "laterality": "", "duration": ""}
  ]
}"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f'Extract the chief complaint from:\n"""\n{clinical_text}\n"""'}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=300, do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return parse_chief_complaint_json(response)


## Module 2 — Patient History Extraction

In [4]:
def parse_history_json(response):
    default = {
        "history_present_illness": [],
        "past_medical_surgical": [],
        "family_history": [],
        "social_history": []
    }
    try:
        data = json.loads(response)
    except json.JSONDecodeError:
        match = re.search(r'\{[\s\S]*\}', response)
        if not match:
            return default
        try:
            data = json.loads(match.group())
        except json.JSONDecodeError:
            return default

    result = {}
    for bucket in default:
        values = data.get(bucket, [])
        if not isinstance(values, list):
            values = [values]
        result[bucket] = [str(x).strip() for x in values if x is not None and str(x).strip()]
    return result


def extract_patient_history(clinical_text):
    system_prompt = """You are an ENT clinical documentation assistant.

Extract ONLY patient HISTORY. Do not include chief complaint or examination findings.

Buckets:
- history_present_illness: how/why current symptoms developed, recurrence, triggers, progression.
- past_medical_surgical: prior diagnoses, ENT surgeries, operations, comorbidities.
- family_history: only if explicitly mentioned.
- social_history: smoking, alcohol, occupational noise/dust exposure.

Rules:
- Do not infer missing information.
- Preserve explicitly documented negative history such as "no history of diabetes".
- If a bucket has no data, return [].
- Do not include diagnosis, examination findings, or investigations.
- Return valid JSON only.

Schema:
{
  "history_present_illness": [],
  "past_medical_surgical": [],
  "family_history": [],
  "social_history": []
}"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f'Extract patient history from:\n"""\n{clinical_text}\n"""'}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=500, do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return parse_history_json(response)


## Module 3 — Examination Findings Extraction

In [5]:
def parse_examination_json(response):
    default = {
        "ear_otoscopy": [],
        "nose_rhinoscopy": [],
        "throat_oral_cavity": [],
        "larynx": [],
        "neck_lymph_nodes": [],
        "other": []
    }

    try:
        data = json.loads(response)
    except json.JSONDecodeError:
        match = re.search(r'\{[\s\S]*\}', response)
        if not match:
            return default
        try:
            data = json.loads(match.group())
        except json.JSONDecodeError:
            return default

    result = {k: [] for k in default}

    for category in ["ear_otoscopy", "nose_rhinoscopy"]:
        values = data.get(category, [])
        if not isinstance(values, list):
            values = [values]
        for item in values:
            if isinstance(item, dict):
                side = str(item.get("side", "Not specified")).strip() or "Not specified"
                finding = str(item.get("finding", "")).strip()
                if finding:
                    result[category].append({"side": side, "finding": finding})

    for category in ["throat_oral_cavity", "larynx", "neck_lymph_nodes", "other"]:
        values = data.get(category, [])
        if not isinstance(values, list):
            values = [values]
        for item in values:
            if isinstance(item, dict):
                item = item.get("finding", "")
            item = str(item).strip()
            if item:
                result[category].append(item)

    return result


def extract_examination_findings(clinical_text):
    system_prompt = """You are an ENT clinical documentation assistant.

Extract ONLY EXAMINATION FINDINGS and map them to:
ear_otoscopy, nose_rhinoscopy, throat_oral_cavity, larynx, neck_lymph_nodes, other.

Rules:
- Record laterality when given.
- Ear and nose entries use {"side": "", "finding": ""}.
- Do not include chief complaint, history, investigations, or diagnosis.
- Do not upgrade findings into diagnoses. For example, "perforation" stays "perforation".
- If a category is not documented, return [].
- Do not infer findings.
- Return valid JSON only.

Schema:
{
  "ear_otoscopy": [{"side": "", "finding": ""}],
  "nose_rhinoscopy": [{"side": "", "finding": ""}],
  "throat_oral_cavity": [],
  "larynx": [],
  "neck_lymph_nodes": [],
  "other": []
}"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f'Extract examination findings from:\n"""\n{clinical_text}\n"""'}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=500, do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return parse_examination_json(response)


## Module 4 — Diagnosis Extraction

In [6]:
def parse_diagnosis_json(response):
    default = {"primary_diagnosis": None, "differential_diagnosis": []}

    try:
        data = json.loads(response)
    except json.JSONDecodeError:
        match = re.search(r'\{[\s\S]*\}', response)
        if not match:
            return default
        try:
            data = json.loads(match.group())
        except json.JSONDecodeError:
            return default

    primary = data.get("primary_diagnosis")
    if primary is None:
        primary_result = None
    elif isinstance(primary, dict):
        primary_result = {
            "condition": str(primary.get("condition", "")).strip(),
            "laterality": str(primary.get("laterality", "")).strip(),
            "subtype": str(primary.get("subtype", "")).strip()
        }
    else:
        primary_result = None

    differential = data.get("differential_diagnosis", [])
    if not isinstance(differential, list):
        differential = [differential]
    differential = [
        str(x.get("condition", "") if isinstance(x, dict) else x).strip()
        for x in differential
    ]
    differential = [x for x in differential if x]

    return {
        "primary_diagnosis": primary_result,
        "differential_diagnosis": differential
    }


def extract_diagnosis(clinical_text):
    system_prompt = """You are an ENT clinical documentation assistant.

Extract ONLY DIAGNOSIS information.

Rules:
- primary_diagnosis = main STATED diagnosis, including laterality/subtype if explicitly given.
- differential_diagnosis = only if explicitly framed as a possibility/differential, such as "likely", "?", "rule out", "r/o", "differential includes", "possible", "suspected".
- Never generate a diagnosis from symptoms or findings.
- If no primary diagnosis is explicitly stated, primary_diagnosis = null.
- Preserve ENT terminology/abbreviations as written, e.g. CSOM, OSA, CRS.
- Do not include complaints, history, examination findings, investigations, or treatment.
- Return valid JSON only.

Schema:
{
  "primary_diagnosis": {"condition": "", "laterality": "", "subtype": ""},
  "differential_diagnosis": []
}"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f'Extract diagnosis from:\n"""\n{clinical_text}\n"""'}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=400, do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return parse_diagnosis_json(response)


## Module 5 — Investigations Extraction

In [7]:
def parse_investigation_json(response):
    default = {"investigations": []}

    try:
        data = json.loads(response)
    except json.JSONDecodeError:
        match = re.search(r'\{[\s\S]*\}', response)
        if not match:
            return default
        try:
            data = json.loads(match.group())
        except json.JSONDecodeError:
            return default

    values = data.get("investigations", [])
    if not isinstance(values, list):
        values = [values]

    cleaned = []
    for item in values:
        if not isinstance(item, dict):
            continue

        test = str(item.get("test", "")).strip()
        status = str(item.get("status", "")).strip().lower()
        result = item.get("result", None)

        if result is not None:
            result = str(result).strip() or None

        if status not in ["ordered", "result available"]:
            status = "result available" if result is not None else "ordered"

        if test:
            cleaned.append({
                "test": test,
                "status": status,
                "result": result
            })

    return {"investigations": cleaned}


def extract_investigations(clinical_text):
    system_prompt = """You are an ENT clinical documentation assistant.

Extract ONLY INVESTIGATIONS.

For each investigation capture:
- test
- status: exactly "ordered" or "result available"
- result: documented result/finding, or null

Rules:
- If only the test name is mentioned with no result: status = "ordered", result = null.
- If a result/finding is documented: status = "result available".
- Do not infer results.
- Do not include symptoms, history, examination findings unless they are explicitly reported as a test result, diagnosis, or treatment.
- Common ENT tests include PTA, tympanometry, swab culture, CT/MRI PNS or temporal bone, nasal endoscopy, biopsy, allergy testing.
- Preserve numerical values and documented result wording.
- Return valid JSON only.

Schema:
{
  "investigations": [
    {"test": "", "status": "", "result": null}
  ]
}"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f'Extract investigations from:\n"""\n{clinical_text}\n"""'}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=500, do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return parse_investigation_json(response)


## Final Integration — Run All 5 Modules

In [8]:
def extract_ent_note(clinical_text):
    """
    Run all five ENT extraction modules and return one combined JSON object.
    """

    return {
        "chief_complaints": extract_chief_complaint(clinical_text)["chief_complaints"],
        "history": extract_patient_history(clinical_text),
        "examination": extract_examination_findings(clinical_text),
        "diagnosis": extract_diagnosis(clinical_text),
        "investigations": extract_investigations(clinical_text)["investigations"]
    }


## Test the Complete Pipeline

In [9]:
clinical_text = """
45M presents with right-sided ear pain and discharge for 5 days.
He reports recurrent episodes after swimming and symptoms have gradually worsened.

He underwent right ear surgery 5 years ago. He has hypertension and no history of diabetes.
His father had hearing loss. He smokes 5 cigarettes per day and occasionally drinks alcohol.

On examination, the right ear shows a central tympanic membrane perforation with discharge.
The left tympanic membrane is intact. Anterior rhinoscopy shows a deviated septum to the left.

Pure tone audiometry showed mild conductive hearing loss in the right ear.
Tympanometry was ordered for both ears.

The final diagnosis is right CSOM, tubotympanic type.
Cholesteatoma is considered as a possible differential diagnosis.
"""

result = extract_ent_note(clinical_text)

print(json.dumps(result, indent=2, ensure_ascii=False))


{
  "chief_complaints": [
    {
      "symptom": "right-sided ear pain",
      "laterality": "right",
      "duration": "5 days"
    }
  ],
  "history": {
    "history_present_illness": [
      "{'symptoms': 'right-sided ear pain and discharge', 'duration': '5 days', 'recurrence': 'after swimming', 'progression': 'gradually worsening'}"
    ],
    "past_medical_surgical": [
      "{'diagnosis': 'right ear surgery 5 years ago', 'surgery_type': 'tympanoplasty'}",
      "{'comorbidity': 'hypertension'}"
    ],
    "family_history": [
      "{'condition': 'hearing loss in his father'}"
    ],
    "social_history": [
      "{'smoking_status': 'smokes 5 cigarettes per day', 'alcohol_consumption': 'occasionally drinks alcohol'}"
    ]
  },
  "examination": {
    "ear_otoscopy": [
      {
        "side": "right",
        "finding": "central tympanic membrane perforation with discharge"
      }
    ],
    "nose_rhinoscopy": [
      {
        "side": "Not specified",
        "finding": "deviated

## Interactive Clinical Note Input

In [10]:
clinical_text = input("Paste ENT clinical note:\n\n")

result = extract_ent_note(clinical_text)

print("\n===== FINAL ENT EXTRACTION =====\n")
print(json.dumps(result, indent=2, ensure_ascii=False))


Paste ENT clinical note:

45-year-old male presents with right ear pain and ear discharge for 5 days. He reports recurrent episodes of ear discharge after swimming, with symptoms gradually worsening.  He underwent right ear surgery 5 years ago. He has a history of hypertension and no history of diabetes. His father had a history of hearing loss. He smokes 5 cigarettes per day and occasionally consumes alcohol. He works in a noisy industrial environment.  On examination, the right ear shows a central tympanic membrane perforation with minimal discharge. The left tympanic membrane is intact. Anterior rhinoscopy shows a deviated nasal septum to the left. Oral cavity is normal. Both vocal cords are mobile. No cervical lymph nodes are palpable.  Pure tone audiometry showed mild conductive hearing loss in the right ear. Tympanometry was ordered for both ears. CT temporal bone was advised.  The final diagnosis is right CSOM, tubotympanic type. Cholesteatoma is considered as a possible differe